In [1]:
!pip uninstall -y torchvision -q
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.3 MB/s eta 0:00:00


In [2]:
import torch
import json

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("Model:", MODEL_NAME)

Device: cuda


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-1.5B-Instruct


In [16]:
def generate_json(text, max_new_tokens=200):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an industrial information extraction system. "
                "Return only valid JSON. "
                "Do not add explanations before or after the JSON."
            )
        },
        {
            "role": "user",
            "content": f"""
Aşağıdaki teknik metni analiz et.

METİN:
{text}

Şu JSON yapısında cevap ver:

{{
    "equipment": "",
    "symptoms": [],
    "possible_fault": "",
    "severity": ""
}}

Kurallar:
- Sadece geçerli JSON üret.
- JSON dışında açıklama yazma.
- symptoms bir liste olmalı.
- severity yalnızca "low", "medium" veya "high" olabilir.
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    result = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Markdown kod bloğu temizliği
    clean_result = result

    if clean_result.startswith("```json"):
        clean_result = clean_result[7:]

    elif clean_result.startswith("```"):
        clean_result = clean_result[3:]

    if clean_result.endswith("```"):
        clean_result = clean_result[:-3]

    clean_result = clean_result.strip()

    try:

        data = json.loads(clean_result)

        return data

    except json.JSONDecodeError as error:

        print("JSON ayrıştırılamadı.")
        print("Model çıktısı:")
        print(result)

        print("\nHata:")
        print(error)

        return None

In [17]:
text = """
Bir elektrik motorunda normalden yüksek titreşim ve
çalışma sıcaklığında artış gözlemlendi.
Bakım ekibi rulman bölgesinden anormal ses geldiğini de bildirdi.
"""

data = generate_json(text)

print(data)

{'equipment': 'Motor', 'symptoms': ['Normal titreşim ve çalışma sıcaklığı artırdı', 'Anormal ses geldi'], 'possible_fault': 'Elektrik motorunun titreşim durumu değişti veya bir diğer elektronik malzeme problemi var', 'severity': 'high'}


In [18]:
print(type(data))

<class 'dict'>


In [19]:
def validate_data(data):

    if data is None:
        return False

    required_fields = [
        "equipment",
        "symptoms",
        "possible_fault",
        "severity"
    ]

    for field in required_fields:

        if field not in data:
            print("Eksik alan:", field)
            return False

    if not isinstance(data["symptoms"], list):
        print("symptoms liste olmalı.")
        return False

    allowed_severity = [
        "low",
        "medium",
        "high"
    ]

    if data["severity"] not in allowed_severity:
        print(
            "Geçersiz severity:",
            data["severity"]
        )
        return False

    return True

In [20]:
if validate_data(data):

    print("JSON yapısı geçerli.")

else:

    print("JSON yapısı hatalı.")

JSON yapısı geçerli.
